In [ ]:
import re
import numpy as np

from datetime import datetime
from osgeo import gdal
from pathlib import Path

gdal.UseExceptions()
gdal.SetConfigOption("GDAL_NUM_THREADS", "ALL_CPUS")

In [ ]:
YEAR = 2024
REWRITE = False

In [ ]:
SOURCE_ROOT = Path("/nvm9/data/cuBoulder/v2026")
YEAR_DIR = SOURCE_ROOT / str(YEAR - 1)
YEAR_DIR_END = SOURCE_ROOT / str(YEAR)
OUT_NETCDF = Path(f"/perc10/tmp/wy{YEAR}.nc")

In [ ]:
if OUT_NETCDF.exists() and REWRITE:
    OUT_NETCDF.unlink()

In [ ]:
year_files = sorted(list(YEAR_DIR.glob(f"*_{YEAR - 1}1*.tif")))
year_files += sorted(list(YEAR_DIR_END.glob(f"*_{YEAR}0*.tif")))

In [ ]:
def get_day(filename):
    match = re.search(r"\d{8}", filename)
    return (
        (datetime.strptime(match.group(), "%Y%m%d") - datetime(1970, 1, 1)).days
        if match
        else 0
    )


dates = [get_day(f.name) for f in year_files]

### Get Metadata from first file 

In [ ]:
src_ds = gdal.Open(year_files[0])
width, height = src_ds.RasterXSize, src_ds.RasterYSize
gt = src_ds.GetGeoTransform()
srs_wkt = src_ds.GetProjection()
dtype = src_ds.GetRasterBand(1).DataType

x_coords = gt[0] + (np.arange(width) + 0.5) * gt[1]
y_coords = gt[3] + (np.arange(height) + 0.5) * gt[5]

src_ds = None

### Define structure

In [ ]:
netcdf_options = [
    "FORMAT=NC4",
]

In [ ]:
driver = gdal.GetDriverByName("netCDF")
netcdf_ds = driver.CreateMultiDimensional(str(OUT_NETCDF), [], netcdf_options)
root_group = netcdf_ds.GetRootGroup()

In [ ]:
dim_time = root_group.CreateDimension(
    "time", "TEMPORAL", "DATE", len(year_files)
)
dim_y = root_group.CreateDimension("lat", "HORIZONTAL_Y", "NORTH", height)
dim_x = root_group.CreateDimension("lon", "HORIZONTAL_X", "EAST", width)

### Init arrays

In [ ]:
nodata_val = -9999.0
array_options = [
    "COMPRESS=DEFLATE",
    "ZLEVEL=4",
    "CHUNKING_STRATEGY=CUSTOM",
    "BLOCKSIZE=10,512,512",
]

In [ ]:
time_array = root_group.CreateMDArray(
    "time", [dim_time], gdal.ExtendedDataType.Create(gdal.GDT_Int32)
)
y_array = root_group.CreateMDArray(
    "lat", [dim_y], gdal.ExtendedDataType.Create(gdal.GDT_Float64)
)
x_array = root_group.CreateMDArray(
    "lon", [dim_x], gdal.ExtendedDataType.Create(gdal.GDT_Float64)
)
crs_array = root_group.CreateMDArray(
    "crs", [], gdal.ExtendedDataType.Create(gdal.GDT_Int32)
)
data_array = root_group.CreateMDArray(
    "SWE",
    [dim_time, dim_y, dim_x],
    gdal.ExtendedDataType.Create(dtype),
    array_options,
)

time_array.SetNoDataValue(0)
y_array.SetNoDataValue(0.0)
x_array.SetNoDataValue(0.0)
data_array.SetNoDataValue(nodata_val)

### Attributes

In [ ]:
str_type = gdal.ExtendedDataType.CreateString()
int_type = gdal.ExtendedDataType.Create(gdal.GDT_Int32)
coord_type = gdal.ExtendedDataType.Create(gdal.GDT_Float64)
data_type = gdal.ExtendedDataType.Create(dtype)

root_group.CreateAttribute("Conventions", [], str_type).Write("CF-1.8")
root_group.CreateAttribute("title", [], str_type).Write("CU Boulder SWE")

# Dim Attributes
time_array.CreateAttribute("_FillValue", [], int_type).Write(0)
time_array.CreateAttribute("axis", [], str_type).Write("T")
time_array.CreateAttribute("long_name", [], str_type).Write("time")
time_array.CreateAttribute("units", [], str_type).Write(
    "days since 1970-01-01 00:00:00"
)
time_array.CreateAttribute("calendar", [], str_type).Write(
    "proleptic_gregorian"
)

y_array.CreateAttribute("_FillValue", [], coord_type).Write(0.0)
y_array.CreateAttribute("units", [], str_type).Write("degrees")
y_array.CreateAttribute("long_name", [], str_type).Write("latitude")
y_array.CreateAttribute("axis", [], str_type).Write("y")

x_array.CreateAttribute("_FillValue", [], coord_type).Write(0.0)
x_array.CreateAttribute("units", [], str_type).Write("degrees")
x_array.CreateAttribute("long_name", [], str_type).Write("longitude")
x_array.CreateAttribute("axis", [], str_type).Write("x")

# CRS
crs_array.CreateAttribute("grid_mapping_name", [], str_type).Write(
    "latitude_longitude"
)
crs_array.CreateAttribute("crs_wkt", [], str_type).Write(srs_wkt)
crs_array.CreateAttribute("spatial_ref", [], str_type).Write(srs_wkt)

# SWE Attributes
data_array.CreateAttribute("units", [], str_type).Write("meters")
data_array.CreateAttribute("long_name", [], str_type).Write(
    "Snow Water Equivalent"
)
data_array.CreateAttribute("grid_mapping", [], str_type).Write("crs")
data_array.CreateAttribute("_FillValue", [], data_type).Write(nodata_val)

### Data

In [ ]:
# Coordinates and CRS
crs_array.Write(np.int32(0))
time_array.Write(np.array(dates, dtype="int32"))
y_array.Write(y_coords.astype("float64"))
x_array.Write(x_coords.astype("float64"))

Write the data and set all values less than 0 to -9999, which is set as `no_data` on the file

The "stock" value from CuB is `-3.4028235e+38`, which can cause floating point parsing issues upon loading the file.
These will propagate upon calculating SWE statistics when NaN values are not properly set.

In [ ]:
batch_size = 10

for batch_idx in range(0, len(year_files), batch_size):
    batch_files = year_files[batch_idx : batch_idx + batch_size]

    arrays = []
    for f in batch_files:
        raster_file = gdal.Open(f)
        data = raster_file.GetRasterBand(1).ReadAsArray()
        data[data < 0] = np.nan
        arrays.append(data)
        raster_file = None

    data_array.Write(
        np.stack(arrays),
        array_start_idx=[batch_idx, 0, 0],
        count=[len(batch_files), height, width],
    )

    print("*", end="", flush=True)

In [ ]:
time_array = None
y_array = None
x_array = None
crs_array = None
data_array = None
dim_time = None
dim_y = None
dim_x = None
root_group = None

netcdf_ds.FlushCache()
netcdf_ds = None

### Verify

In [ ]:
import xarray as xr

In [ ]:
nc_file = xr.open_dataset(OUT_NETCDF, engine="h5netcdf")

In [ ]:
nc_file

In [ ]:
nc_file.time[-1]

In [ ]:
nc_file["SWE"].sel(time="2024-05-28").plot()

In [ ]:
nc_file.close()

## Add CBRFC zones

Create a zone mask for the domain
```bash
gdal_rasterize \
  -a gid -init 0 -a_nodata 0 \
  -tr 0.005 0.005 \
  -te -125.0 31.0 -102.0 49.0 \
  -a_srs EPSG:4269 \
  -co "BAND_NAMES=cbrfc_mask" \
  -co "FORMAT=NC4" \
  -ot int32 \
  -sql "SELECT gid, ST_Transform(geom, 4269) as geom FROM cbrfc_zones" \
  PG:"service=swe_db" \
  cu_boulder_swe_cbrfc_mask.nc
```

In [ ]:
import netCDF4 as nc

In [ ]:
CBRFC_MASK = SOURCE_ROOT.parent / "cu_boulder_swe_cbrfc_mask.nc"

In [ ]:
with (
    nc.Dataset(CBRFC_MASK, "r") as mask_file,
    nc.Dataset(OUT_NETCDF, "a") as year_file,
):
    src_var = mask_file.variables["cbrfc_mask"]

    ref_chunks = year_file.variables["SWE"].chunking()[1:]

    mask_variable = year_file.createVariable(
        varname="cbrfc_zone_gid",
        datatype=src_var.datatype,
        dimensions=("lat", "lon"),
        zlib=True,
        complevel=4,
        chunksizes=ref_chunks,
    )

    # Copy attributes and data
    mask_variable.setncatts(
        {k: src_var.getncattr(k) for k in src_var.ncattrs()}
    )
    # Ensure the correct orientation by using flipup()
    mask_variable[:] = np.flipud(src_var[:])